In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
# 2. Load the dataset
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/PyTorch-ML/100_Unique_QA_Dataset.csv')
df.head()

Mounted at /content/drive


,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [2]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [3]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [4]:
# vocab
vocab = {'<UNK>':0}

In [5]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)


In [6]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [7]:
len(vocab)

324

In [8]:
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [9]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

In [11]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [12]:
dataset = QADataset(df, vocab)

In [14]:
dataset[11]

(tensor([10, 55,  3, 56,  5, 57]), tensor([58]))

In [15]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [19]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[ 42,  18,   2,   3, 281,  12,   3, 282]]) tensor([205])
tensor([[  1,   2,   3,  92, 137,  19,   3,  45]]) tensor([185])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([52])
tensor([[  1,   2,   3,   4,   5, 113]]) tensor([114])
tensor([[ 42, 137,   2,  62,  39,   3, 322, 323]]) tensor([6])
tensor([[10, 29,  3, 30, 31]]) tensor([32])
tensor([[ 10, 140,   3, 141, 142,  12, 143,  83,   3, 144]]) tensor([145])
tensor([[ 10,  96,   3, 104, 239]]) tensor([240])
tensor([[  1,   2,   3,  37, 133,   5,  26]]) tensor([134])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([9])
tensor([[  1,   2,   3, 212,   5,  14, 213, 214]]) tensor([215])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([85])
tensor([[  1,   2,   3, 234,   5, 235]]) tensor([131])
tensor([[  1,   2,   3, 103,   5, 104,  19, 105]]) tensor([106])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([74])
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([321])
tensor([[ 10,  75,   3, 296,  19, 297]]) tensor([298])
tensor([[ 42, 26

In [17]:
import torch.nn as nn

In [20]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)# embedding vector dimension
    self.rnn = nn.RNN(50, 64, batch_first=True) #hidden layer
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [21]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [33]:
learning_rate = 0.001
epochs = 100

In [34]:
model = SimpleRNN(len(vocab))

In [35]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [36]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 520.842483
Epoch: 2, Loss: 453.590252
Epoch: 3, Loss: 374.306961
Epoch: 4, Loss: 310.367391
Epoch: 5, Loss: 258.214609
Epoch: 6, Loss: 210.425619
Epoch: 7, Loss: 166.980895
Epoch: 8, Loss: 130.322501
Epoch: 9, Loss: 100.016994
Epoch: 10, Loss: 76.361032
Epoch: 11, Loss: 59.183023
Epoch: 12, Loss: 46.626629
Epoch: 13, Loss: 36.964410
Epoch: 14, Loss: 30.147815
Epoch: 15, Loss: 24.729000
Epoch: 16, Loss: 20.624217
Epoch: 17, Loss: 17.519782
Epoch: 18, Loss: 14.956267
Epoch: 19, Loss: 12.655445
Epoch: 20, Loss: 11.155257
Epoch: 21, Loss: 9.695066
Epoch: 22, Loss: 8.540596
Epoch: 23, Loss: 7.569426
Epoch: 24, Loss: 6.734934
Epoch: 25, Loss: 6.058316
Epoch: 26, Loss: 5.473497
Epoch: 27, Loss: 4.932416
Epoch: 28, Loss: 4.479962
Epoch: 29, Loss: 4.091656
Epoch: 30, Loss: 3.742635
Epoch: 31, Loss: 3.443761
Epoch: 32, Loss: 3.161270
Epoch: 33, Loss: 2.924048
Epoch: 34, Loss: 2.702175
Epoch: 35, Loss: 2.503279
Epoch: 36, Loss: 2.327467
Epoch: 37, Loss: 2.166081
Epoch: 38, Loss: 2

In [26]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [27]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [37]:
list(vocab.keys())[7]

'paris'